# DL Streamer: car detection and color classification (teaching notebook)

This notebook targets **Linux** (same layout as `utils.py`). The clip is **`Videos/1900-151662242_medium.mp4`** next to this notebook (see `DEFAULT_VIDEO` in `utils.py`). IR models use **`../1/models/INT8/`** (`Detection/model.xml` and `Classification/model.xml`). Run Jupyter with working directory set to this folder (`2/`) so these relative paths resolve. If `decodebin3` is missing, set `GST_DECODEBIN=decodebin`.


## 1. Imports

`run_pipeline` only runs a pipeline string with `gst-launch-1.0` (it does not build the string).


In [ ]:
import os
import subprocess


def run_pipeline(pipeline: str) -> None:
    """Execute a pipeline string with gst-launch-1.0 (does not build the string)."""
    pipeline = pipeline.strip()
    r = subprocess.run(
        ["gst-launch-1.0", pipeline],
        env={**os.environ, "GST_DEBUG": "0", "GST_DEBUG_NO_COLOR": "1"},
        check=False,
        stderr=subprocess.PIPE,
        text=True,
    )
    if r.returncode != 0:
        if r.stderr and r.stderr.strip():
            print("--- gst-launch stderr ---")
            print(r.stderr.strip()[-8000:])
        print(f"(gst-launch exited with code {r.returncode}; closing the video window often causes nonzero.)")
    print("Pipeline ended.")


## 2. Display video (no inference)

Playback only: **decode** and **display**. No DL Streamer inference elements.


```mermaid
flowchart LR
  V[Video file] --> F[filesrc]
  F --> D[decodebin3]
  D --> C[videoconvert]
  C --> S[autovideosink]
```


In [ ]:
video_path = "Videos/1900-151662242_medium.mp4"

decodebin_element = os.environ.get("GST_DECODEBIN", "decodebin3")

pipeline = f"""
filesrc location={video_path} !
{decodebin_element} !
videoconvert !
autovideosink sync=true
"""
pipeline = " ".join(line.strip() for line in pipeline.splitlines() if line.strip())

print(pipeline)
run_pipeline(pipeline)


## 3. Detection pipeline

`gvadetect` runs the detector IR; **`gvafpscounter`** is a separate FPS overlay element. **`gvawatermark`** draws metadata on the frame.


```mermaid
flowchart LR
  V[Video file] --> F[filesrc]
  F --> D[decodebin3]
  D --> Det["gvadetect<br/>(Detection)"]
  Det --> W[gvawatermark]
  W --> FPS["gvafpscounter<br/>(FPS)"]
  FPS --> C[videoconvert]
  C --> S[autovideosink]

  style Det fill:#ffcccc,stroke:#d32f2f,stroke-width:3px
  style FPS fill:#ffcccc,stroke:#d32f2f,stroke-width:3px
```


In [ ]:
video_path = "Videos/1900-151662242_medium.mp4"
detection_model_path = "../1/models/INT8/Detection/model.xml"
detection_device = "GPU"

print("Detection model:", detection_model_path)
print("Device:", detection_device)

decodebin_element = os.environ.get("GST_DECODEBIN", "decodebin3")

pipeline = f"""
filesrc location={video_path} !
{decodebin_element} !
gvadetect model={detection_model_path} device={detection_device} pre-process-backend=opencv !
gvawatermark !
gvafpscounter !
videoconvert !
autovideosink sync=true
"""
pipeline = " ".join(line.strip() for line in pipeline.splitlines() if line.strip())

print(pipeline)
run_pipeline(pipeline)


## 4. Detection + classification pipeline

After detection, **`gvatrack`** associates ROIs across frames; **`gvaclassify`** runs the color (or attribute) classifier. `reclassify-interval` controls how often classification runs.


```mermaid
flowchart LR
  V[Video file] --> F[filesrc]
  F --> D[decodebin3]
  D --> Det[gvadetect]
  Det --> T[gvatrack]
  T --> Cls[gvaclassify]
  Cls --> Q[queue]
  Q --> W[gvawatermark]
  W --> FPS[gvafpscounter]
  FPS --> C[videoconvert]
  C --> S[autovideosink]
```


In [ ]:
video_path = "Videos/1900-151662242_medium.mp4"
detection_model_path = "../1/models/INT8/Detection/model.xml"
classification_model_path = "../1/models/INT8/Classification/model.xml"
detection_device = "GPU"
classification_device = "CPU"
reclassify_interval = 2

print("Detection model:", detection_model_path)
print("Detection device:", detection_device)
print("Classification model:", classification_model_path)
print("Classification device:", classification_device)

decodebin_element = os.environ.get("GST_DECODEBIN", "decodebin3")

pipeline = f"""
filesrc location={video_path} !
{decodebin_element} !
gvadetect model={detection_model_path} device={detection_device} pre-process-backend=opencv !
gvatrack !
gvaclassify model={classification_model_path} device={classification_device} pre-process-backend=opencv reclassify-interval={reclassify_interval} !
queue ! gvawatermark ! gvafpscounter !
videoconvert !
autovideosink sync=true
"""
pipeline = " ".join(line.strip() for line in pipeline.splitlines() if line.strip())

print(pipeline)
run_pipeline(pipeline)


## 5. Benchmarking: one pipeline vs duplicated components

**Scaling** here means **repeating** processing blocks in the same `gst-launch` line (e.g. a second `gvadetect` after `queue`). There is no hidden helper — compare `pipeline_1` and `pipeline_2` directly.

Below, **`fakesink`** avoids opening a window; use `autovideosink` if you want to watch both stages during debugging.


```mermaid
flowchart LR
  subgraph one [pipeline_1]
    A1[filesrc] --> D1[decode] --> G1[gvadetect] --> F1[FPS] --> K1[fakesink]
  end
  subgraph two [pipeline_2]
    A2[filesrc] --> D2[decode] --> G2a[gvadetect] --> Q[queue] --> G2b[gvadetect] --> F2[FPS] --> K2[fakesink]
  end
```


In [ ]:
video_path = "Videos/1900-151662242_medium.mp4"
detection_model_path = "../1/models/INT8/Detection/model.xml"
detection_device = "GPU"

decodebin_element = os.environ.get("GST_DECODEBIN", "decodebin3")

pipeline_1 = f"""
filesrc location={video_path} !
{decodebin_element} !
gvadetect model={detection_model_path} device={detection_device} pre-process-backend=opencv !
gvafpscounter !
videoconvert !
fakesink sync=false
"""
pipeline_1 = " ".join(line.strip() for line in pipeline_1.splitlines() if line.strip())

print("--- pipeline_1 (single gvadetect) ---")
print(pipeline_1)
run_pipeline(pipeline_1)


In [ ]:
video_path = "Videos/1900-151662242_medium.mp4"
detection_model_path = "../1/models/INT8/Detection/model.xml"
detection_device = "GPU"

decodebin_element = os.environ.get("GST_DECODEBIN", "decodebin3")

pipeline_2 = f"""
filesrc location={video_path} !
{decodebin_element} !
gvadetect model={detection_model_path} device={detection_device} pre-process-backend=opencv !
queue !
gvadetect model={detection_model_path} device={detection_device} pre-process-backend=opencv !
gvafpscounter !
videoconvert !
fakesink sync=false
"""
pipeline_2 = " ".join(line.strip() for line in pipeline_2.splitlines() if line.strip())

print("--- pipeline_2 (two gvadetect blocks — duplicated component) ---")
print(pipeline_2)
run_pipeline(pipeline_2)
